In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
# Cấu hình Confluent Kafka
KAFKA_SERVER = "YOUR_BOOTSTRAP_SERVER"
KAFKA_API_KEY = "YOUR_API_KEY"
KAFKA_API_SECRET = "YOUR_API_SECRET"
KAFKA_TOPIC = "binance-kline-1m"

In [0]:
kline_volume_path = "/Volumes/workspace/bronze/binance_raw/kline_1m"
kline_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/kline"

## WRITE TO BRONZE (KLINE)

In [0]:
df_kline_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(kline_volume_path)
)

# Add metadatas
df_bronze_kline = df_kline_stream \
    .withColumnRenamed("value", "raw_payload") \
    .withColumn("ingested_at", current_timestamp())

# Write to kline Table
query_kline = (df_bronze_kline.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", kline_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.bronze.kline") 
)

query_kline.awaitTermination()


In [0]:
%sql
-- SELECT * FROM workspace.bronze.kline LIMIT 10;